# Example workflow part 2

In this case we only need one more class (at least on subject level). So the Base_project class is a bit unnecessary. But imagine different data types or classes for different analysis and it all makes sense.
We'll now inherit from the class and add the analysis class on top of this:

In [1]:
import numpy as np
from scipy import stats
import os

from Base_project import Base_project

There's one more trick. Remember how we ran `list_subjects` in the `__init__ ` method of Base_project? We're overwriting the `__init__`method in the new class. So that doesn't get called by default, but it would be nice. We can call the `__init__`-method of the superclass (the one we inherit) from like this:

In [2]:
class Subject( Base_project):
    
    def __init__(self,subID):
        super().__init__();
        self.subID = subID;

In [3]:
s = Subject(2);
print( s.subjects)

['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's09', 's10', 's11', 's12', 's13', 's14', 's15']


Looking at the assignment we can already think about a few methods we will need:

In [4]:
class Subject( Base_project):
    
    def __init__(self,subID):
        super().__init__();
        self.subID = subID;
        
    #we will have to load the data
    def load_data(self):
        pass
    
    #we will use a linear regression
    def run_regression(self):
        pass
    
    #we have to convert the sequence of foodIDs into a 
    def convert_seq(self):
        pass
    
    #a function that plots reaction times against ratings
    def plot(self):
        pass    

Let's start with the `__init__`-method. It should raise an error if the subject doesn't exist. This information is already in the object thanks to the `super().__init__()` -part. So we can just check like this:

In [5]:
subID = 10
s.subject_exists( subID)

True

and in the init method:

In [6]:
class Subject( Base_project):
    
    def __init__(self,subID):
        super().__init__();
        self.subID = subID;
        if not self.subject_exists(subID):
            raise Exception('No such subject in the directory ' + self.path)
        

In [7]:
s = Subject(99)

Exception: No such subject in the directory C:/Users/neugebauer/Documents/Python_workshop/toy_data/

okay. The obvious first step is to load the data into the object. The data is in .csv-files. You would now google for a way to read those into python. You could add `numpy` or `pandas` in your search because you know you will use these packages to handle the data. I usually would use pandas for this, but we'll stick to numpy for now.

You will somehow find out that `np.genfromtxt` is probably what you're looking for.

In [8]:
np.genfromtxt?

Signature:
np.genfromtxt(
    ['fname', "dtype=<class 'float'>", "comments='#'", 'delimiter=None', 'skip_header=0', 'skip_footer=0', 'converters=None', 'missing_values=None', 'filling_values=None', 'usecols=None', 'names=None', 'excludelist=None', 'deletechars=None', "replace_space='_'", 'autostrip=False', 'case_sensitive=True', "defaultfmt='f%i'", 'unpack=None', 'usemask=False', 'loose=True', 'invalid_raise=True', 'max_rows=None', "encoding='bytes'"],
)
Docstring:
Load data from a text file, with missing values handled as specified.

Each line past the first `skip_header` lines is split at the `delimiter`
character, and characters following the `comments` character are discarded.

Parameters
----------
fname : file, str, pathlib.Path, list of str, generator
    File, filename, list, or generator to read.  If the filename
    extension is `.gz` or `.bz2`, the file is first decompressed. Note
    that generators must return byte strings in Python 3k.  The strings
    in a list or produc

You see that there's a whole lot of options here. In our case the files are comma separated, so let's just try it out in the simplest way:

In [9]:
s = Subject(1);
pth = s. path_to_data(1,'food');
print(pth)

C:/Users/neugebauer/Documents/Python_workshop/toy_data/s01\food_ratings.csv


In [10]:
test = np.genfromtxt(pth, delimiter = ',');
print(test)

[[ 1.  2.]
 [ 2.  9.]
 [ 3.  7.]
 [ 4.  4.]
 [ 5. 18.]
 [ 6.  9.]
 [ 7.  6.]
 [ 8. 13.]
 [ 9. 17.]
 [10.  7.]
 [11. 16.]
 [12.  7.]
 [13. 13.]
 [14.  3.]
 [15.  5.]
 [16. 20.]
 [17. 12.]
 [18.  2.]
 [19.  9.]
 [20. 11.]
 [21. 20.]
 [22. 13.]
 [23.  7.]
 [24.  3.]
 [25.  5.]
 [26. 11.]
 [27. 14.]
 [28. 16.]
 [29.  5.]
 [30.  7.]]


That looks good. In readcion times there are floats and integers, that might be harder...

In [11]:
pth2 = s.path_to_data(1,'RT');

In [12]:
t2 = np.genfromtxt( pth2 );
print(t2)

[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan]


Yeah, that looks bad. Either, you try to work around it, or you hope the pandas people have done this for you and try again using pandas:


In [23]:
import pandas as pd
t2 = pd.read_csv(pth2);
t2.head()

,29.0,0.3874505638951348
0,19.0,0.484971
1,7.0,0.531008
2,11.0,0.411700
3,7.0,0.256366
4,24.0,0.555504


Unfortunately, this steals the first row and uses it as names for columns. There sure is a way around that:

In [43]:
pd.read_csv?

Signature:
pd.read_csv(
    ['filepath_or_buffer', "sep=','", 'delimiter=None', "header='infer'", 'names=None', 'index_col=None', 'usecols=None', 'squeeze=False', 'prefix=None', 'mangle_dupe_cols=True', 'dtype=None', 'engine=None', 'converters=None', 'true_values=None', 'false_values=None', 'skipinitialspace=False', 'skiprows=None', 'nrows=None', 'na_values=None', 'keep_default_na=True', 'na_filter=True', 'verbose=False', 'skip_blank_lines=True', 'parse_dates=False', 'infer_datetime_format=False', 'keep_date_col=False', 'date_parser=None', 'dayfirst=False', 'iterator=False', 'chunksize=None', "compression='infer'", 'thousands=None', "decimal=b'.'", 'lineterminator=None', 'quotechar=\'"\'', 'quoting=0', 'escapechar=None', 'comment=None', 'encoding=None', 'dialect=None', 'tupleize_cols=None', 'error_bad_lines=True', 'warn_bad_lines=True', 'skipfooter=0', 'doublequote=True', 'delim_whitespace=False', 'low_memory=True', 'memory_map=False', 'float_precision=None'],
)
Docstring:
Read CSV (co

`header` could be worth a try. Usually passing `None` works in cases like this:

In [45]:
import pandas as pd
t2 = pd.read_csv(pth2, header = None);
t2.head()

,0,1
0,29.0,0.387451
1,19.0,0.484971
2,7.0,0.531008
3,11.0,0.411700
4,7.0,0.256366


Thanks, pandas people! We won't use pandas for the analysis, but for input/output is usually way easier to use than numpy. Let's have a look at the type of what we got back

In [16]:
print(type(t2))

<class 'pandas.core.frame.DataFrame'>


okay, a DataFrame. As usual, we can use `np.array` to convert other objects into arrays. Or we use the fact, that pandas is a wrapper around numpy and we can get the `values` attribute, which is the raw array:

In [24]:
t2_array = t2.values;
print(type(t2_array));
print(t2_array[:10,:])

<class 'numpy.ndarray'>
[[19.          0.48497081]
 [ 7.          0.53100802]
 [11.          0.41169963]
 [ 7.          0.25636602]
 [24.          0.55550408]
 [23.          1.04104525]
 [21.          0.41044408]
 [ 1.          0.4713283 ]
 [ 1.          0.73012736]
 [14.          0.41438151]]


that is a good approach. So for all of the files we will first read them using pandas and then convert them to np.arrays:

In [18]:
def read_data(subject, typ):
    path = subject.path_to_data( subject.subID, typ);
    df = pd.read_csv( path );
    return df.values

In [21]:
array= read_data( s, 'food');
print(array)

[[ 2  9]
 [ 3  7]
 [ 4  4]
 [ 5 18]
 [ 6  9]
 [ 7  6]
 [ 8 13]
 [ 9 17]
 [10  7]
 [11 16]
 [12  7]
 [13 13]
 [14  3]
 [15  5]
 [16 20]
 [17 12]
 [18  2]
 [19  9]
 [20 11]
 [21 20]
 [22 13]
 [23  7]
 [24  3]
 [25  5]
 [26 11]
 [27 14]
 [28 16]
 [29  5]
 [30  7]]


A few google searches and a bit of coding and we might arrive here:

In [46]:
def load_data(subject):
    
    load_rating(subject);
    load_RT(subject);
    
def load_rating(subject):
    
    path = subjects.path_to_data( subjects.subID, 'food' );
    subject.rating = pd.read_csv(path, header = None ).values;
    
def load_RT(subject):
    path = subject.path_to_data( subjects.subID, 'RT' );
    dummy = pd.read_csv( path , header = None ).values;
    subject.RT = dummy[:,1];
    subject.seq = dummy[:,0];

We could also load the rating_seq, but to convert this from the ratings and the sequence is part of the assigment, so we don't. In the class:

In [47]:
class Subject( Base_project):
    
    def __init__(self,subID):
        super().__init__();
        self.subID = subID;
        
    #we will have to load the data
    def load_data(self):
        self.load_rating();
        self.load_RT();
        
    def load_rating(self):
        path = self.path_to_data( self.subID, 'food' );
        self.rating = pd.read_csv(path, header = None).values;
    
    def load_RT(self):
        path = self.path_to_data( self.subID, 'RT' );
        dummy = pd.read_csv( path, header = None ).values;
        self.RT = dummy[:,1]; # second column are RTs
        self.seq = dummy[:,0]; # first column is sequence
    
    #we will use a linear regression
    def run_regression(self):
        pass
    
    #we have to convert the sequence of foodIDs into a 
    def convert_seq(self):
        pass
    
    #a function that plots reaction times against ratings
    def plot(self):
        pass    

In [48]:
s = Subject(1);
s.load_data();
print(s.rating)

[[ 1  2]
 [ 2  9]
 [ 3  7]
 [ 4  4]
 [ 5 18]
 [ 6  9]
 [ 7  6]
 [ 8 13]
 [ 9 17]
 [10  7]
 [11 16]
 [12  7]
 [13 13]
 [14  3]
 [15  5]
 [16 20]
 [17 12]
 [18  2]
 [19  9]
 [20 11]
 [21 20]
 [22 13]
 [23  7]
 [24  3]
 [25  5]
 [26 11]
 [27 14]
 [28 16]
 [29  5]
 [30  7]]


Works fine until here. Trying to convert the sequence of food_ids into a sequence of ratings. You could do something like this:

In [49]:
seq = s.seq;
rate = s.rating

In [55]:
rate_seq = np.zeros(seq.shape)
for i,s in enumerate(seq):
    rating = rate[rate[:,0] == s,1]
    
    rate_seq[i] = rating;
print(rate_seq)  

[ 5.  9.  6. 16.  6.  3.  7. 20.  2.  2.  3.  6. 11.  6. 18.  7.  9. 11.
 20. 17. 17.  4. 13.  7.  2. 18. 13.  4.  2.  5. 13.  9. 16. 12.  5. 18.
 13. 13.  9.  5.  7.  4. 13. 14.  7. 20. 17.  7.  7.  7.  3. 16.  7. 16.
  9.  3. 13. 16.  9.  3.  9. 11.  5.  4. 16.  7. 13.  6. 17. 11.  5.  7.
 12.  5.  7.  7.  7.  9.  2. 14. 12.  3. 20.  7.  9. 11.  3.  4.  7.  5.
  3. 20. 12.  9. 11. 18.  5. 13.  9.  7.  7.  2. 11. 13.  7.  7. 20. 20.
 14.  7.  5.  2.  7. 12. 16. 17. 16.  7.  7.  9. 13. 13.  5. 11.  3. 16.
  5.  2.  3.  2.  5. 11.  5. 13.  2. 16.  9.  5. 14.  9. 20.  9. 20.  7.
 13. 14. 11. 18. 13. 20.]


In [62]:
#but probably list comprehension is quicker
rate_seq = np.array( [rate[ rate[:,0] == s, 1] for s in seq ]);

In [64]:
def convert_seq(subject):
    subject.rate_seq = np.array( [ rate[ rate[:,0] == s, 1] for s in subject.seq ] );

and in the class:

In [ ]:
class Subject( Base_project):
    
    def __init__(self,subID):
        super().__init__();
        self.subID = subID;
        
    #we will have to load the data
    def load_data(self):
        self.load_rating();
        self.load_RT();
        
    def load_rating(self):
        path = self.path_to_data( self.subID, 'food' );
        self.rating = pd.read_csv(path, header = None).values;
    
    def load_RT(self):
        path = self.path_to_data( self.subID, 'RT' );
        dummy = pd.read_csv( path, header = None ).values;
        self.RT = dummy[:,1]; # second column are RTs
        self.seq = dummy[:,0]; # first column is sequence
        
    #we have to convert the sequence of foodIDs into a 
    def convert_seq(self):
        self.rate_seq = np.array( [ rate[ rate[:,0] == s, 1] for s in self.seq ] );
    
    #we will use a linear regression
    def run_regression(self):
        pass
    
    #a function that plots reaction times against ratings
    def plot(self):
        pass    